# OLS IMPLEMENTATION

## Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from LinearRegression import OLS

## Setup

In [2]:
train_test_split_percent = 0.8
y_col_name = "Price"

In [3]:
def fill_null_num_col(frame: pd.DataFrame, val: float | int, cols: [String]):    
    for col_name in cols:
        if col_name not in num_cols:
            continue
            
        frame.fillna({col_name: val}, inplace=True)
    return frame

In [4]:
def fill_null_num_col_median(frame: pd.DataFrame, cols: [String]):    
    for col_name in cols:
        if col_name not in num_cols:
            continue
            
        frame.fillna({col_name: frame[col_name].median()}, inplace=True)
    pass

In [5]:
def fill_null_cat_col(frame: pd.DataFrame, val: String, cols: [String]):
    for col_name in cols:
        if col_name not in cat_cols:
            continue

        frame.fillna({col_name: val}, inplace=True)

In [6]:
def one_hot_encoding(frame: pd.DataFrame, columns: [String] = [], drop_first: bool = False):
    return pd.get_dummies(frame, columns=columns, drop_first=drop_first, dtype=int)

In [7]:
def split_X_y(frame:pd.DataFrame, y: String):
    frame_cols = frame.columns.to_list()
    if y not in frame_cols:
        print(f"Column y: {y} can't be found in dataframe.")
        return None
    return frame.pop(y)

## Data exploration

In [8]:
df = pd.read_csv("./vietnam_housing_dataset.csv")
df_shape = df.shape
df_height = df_shape[0]
df_weidth = df_shape[1]

cols = df.columns.to_list()
num_cols = df.select_dtypes(include="number").columns.to_list()
cat_cols = df.select_dtypes(include=["category", "object", "str"]).columns.to_list()

print(df_shape)
print(f"Numeric columns:\n{num_cols}")
print("\n")
print(f"Categorical columns:\n{cat_cols}")

(30229, 12)
Numeric columns:
['Area', 'Frontage', 'Access Road', 'Floors', 'Bedrooms', 'Bathrooms', 'Price']


Categorical columns:
['Address', 'House direction', 'Balcony direction', 'Legal status', 'Furniture state']


In [9]:
print(df.head())

                                             Address  Area  Frontage  \
0  Dự án The Empire - Vinhomes Ocean Park 2, Xã L...  84.0       NaN   
1  Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...  60.0       NaN   
2  Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...  90.0       6.0   
3  Đường Nguyễn Văn Khối, Phường 11, Gò Vấp, Hồ C...  54.0       NaN   
4   Đường Quang Trung, Phường 8, Gò Vấp, Hồ Chí Minh  92.0       NaN   

   Access Road House direction Balcony direction  Floors  Bedrooms  Bathrooms  \
0          NaN             NaN               NaN     4.0       NaN        NaN   
1          NaN             NaN               NaN     5.0       NaN        NaN   
2         13.0      Đông - Bắc        Đông - Bắc     5.0       NaN        NaN   
3          3.5       Tây - Nam         Tây - Nam     2.0       2.0        3.0   
4          NaN      Đông - Nam        Đông - Nam     2.0       4.0        4.0   

       Legal status Furniture state  Price  
0  Have certificate             NaN

In [10]:
print(f"Null (%) for each column:\n{(df.isnull().mean()*100).round(2)}")

Null (%) for each column:
Address               0.00
Area                  0.00
Frontage             38.25
Access Road          43.99
House direction      70.26
Balcony direction    82.65
Floors               11.92
Bedrooms             17.08
Bathrooms            23.40
Legal status         14.91
Furniture state      46.71
Price                 0.00
dtype: float64


In [11]:
train_test_split_amount = int(df_height*train_test_split_percent)
train = df.iloc[:train_test_split_amount]
test = df.iloc[train_test_split_amount:]

print(train.shape[0])
print(test.shape[0])
print(f"{df_height}:{train.shape[0]+test.shape[0]}")

24183
6046
30229:30229


In [12]:
null_num_cols = df.select_dtypes(include="number").isnull().columns.tolist()
null_cat_cols = df.select_dtypes(include="category").isnull().columns.tolist()

print(f"Numeric columns with null:\n{null_num_cols}")
print("\n")
print(f"Categorical columns with null:\n{null_cat_cols}")

Numeric columns with null:
['Area', 'Frontage', 'Access Road', 'Floors', 'Bedrooms', 'Bathrooms', 'Price']


Categorical columns with null:
[]


## Data processing

Numeric columns:
['Area', 'Frontage', 'Access Road', 'Floors', 'Bedrooms', 'Bathrooms', 'Price']

Categorical columns:
['Address', 'House direction', 'Balcony direction', 'Legal status', 'Furniture state']

### Numerical features

In [13]:
num_col_fill_median = ["Area", "Frontage", "Access Road", "Price"]
num_col_fill_one = ["Floors", "Bedrooms", "Bathrooms"]
print(f"{len(num_col_fill_median+num_col_fill_one)}:{len(num_cols)}")

7:7


In [14]:
fill_null_num_col_median(train, num_col_fill_median)
fill_null_num_col(train, 1, num_col_fill_one)

,Address,Area,Frontage,Access Road,House direction,Balcony direction,Floors,Bedrooms,Bathrooms,Legal status,Furniture state,Price
0,"Dự án The Empire - Vinhomes Ocean Park 2, Xã L...",84.0,4.5,6.0,NaN,NaN,4.0,1.0,1.0,Have certificate,NaN,8.60
1,"Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...",60.0,4.5,6.0,NaN,NaN,5.0,1.0,1.0,NaN,NaN,7.50
2,"Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...",90.0,6.0,13.0,Đông - Bắc,Đông - Bắc,5.0,1.0,1.0,Sale contract,NaN,8.90
3,"Đường Nguyễn Văn Khối, Phường 11, Gò Vấp, Hồ C...",54.0,4.5,3.5,Tây - Nam,Tây - Nam,2.0,2.0,3.0,Have certificate,Full,5.35
4,"Đường Quang Trung, Phường 8, Gò Vấp, Hồ Chí Minh",92.0,4.5,6.0,Đông - Nam,Đông - Nam,2.0,4.0,4.0,Have certificate,Full,6.90
...,...,...,...,...,...,...,...,...,...,...,...,...
24178,"Đường Dương Đình Hội, Phường Phước Long B, Quậ...",60.0,4.0,6.0,Tây - Bắc,NaN,2.0,2.0,2.0,Have certificate,Basic,3.90
24179,"Hẻm 47, Đường Trường Lưu, Phường Long Trường, ...",52.8,8.0,8.0,NaN,NaN,3.0,3.0,3.0,Have certificate,Basic,4.50
24180,"Đường Phạm Văn Đồng, Phường 13, Bình Thạnh, Hồ...",40.2,4.5,6.0,NaN,NaN,7.0,5.0,6.0,NaN,Full,8.90
24181,"Phường Tây Thạnh, Tân Phú, Hồ Chí Minh",40.0,3.5,8.0,NaN,NaN,4.0,4.0,3.0,NaN,NaN,5.40


### Categorical features

In [15]:
print(cat_cols)

['Address', 'House direction', 'Balcony direction', 'Legal status', 'Furniture state']


In [16]:
fill_null_cat_col(train, "Unknown", cat_cols)

X_train = one_hot_encoding(train, cat_cols, True)

display(train)
display(X_train)

,Address,Area,Frontage,Access Road,House direction,Balcony direction,Floors,Bedrooms,Bathrooms,Legal status,Furniture state,Price
0,"Dự án The Empire - Vinhomes Ocean Park 2, Xã L...",84.0,4.5,6.0,Unknown,Unknown,4.0,1.0,1.0,Have certificate,Unknown,8.60
1,"Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...",60.0,4.5,6.0,Unknown,Unknown,5.0,1.0,1.0,Unknown,Unknown,7.50
2,"Dự án The Crown - Vinhomes Ocean Park 3, Xã Ng...",90.0,6.0,13.0,Đông - Bắc,Đông - Bắc,5.0,1.0,1.0,Sale contract,Unknown,8.90
3,"Đường Nguyễn Văn Khối, Phường 11, Gò Vấp, Hồ C...",54.0,4.5,3.5,Tây - Nam,Tây - Nam,2.0,2.0,3.0,Have certificate,Full,5.35
4,"Đường Quang Trung, Phường 8, Gò Vấp, Hồ Chí Minh",92.0,4.5,6.0,Đông - Nam,Đông - Nam,2.0,4.0,4.0,Have certificate,Full,6.90
...,...,...,...,...,...,...,...,...,...,...,...,...
24178,"Đường Dương Đình Hội, Phường Phước Long B, Quậ...",60.0,4.0,6.0,Tây - Bắc,Unknown,2.0,2.0,2.0,Have certificate,Basic,3.90
24179,"Hẻm 47, Đường Trường Lưu, Phường Long Trường, ...",52.8,8.0,8.0,Unknown,Unknown,3.0,3.0,3.0,Have certificate,Basic,4.50
24180,"Đường Phạm Văn Đồng, Phường 13, Bình Thạnh, Hồ...",40.2,4.5,6.0,Unknown,Unknown,7.0,5.0,6.0,Unknown,Full,8.90
24181,"Phường Tây Thạnh, Tân Phú, Hồ Chí Minh",40.0,3.5,8.0,Unknown,Unknown,4.0,4.0,3.0,Unknown,Unknown,5.40


,Area,Frontage,Access Road,Floors,Bedrooms,Bathrooms,Price,"Address_1/ 113, Đường Lâm Thị Hố, Phường Tân Chánh Hiệp, Quận 12, Hồ Chí Minh","Address_1/ Phố Trần Đại Nghĩa, Xã Tân Kiên, Bình Chánh, Hồ Chí Minh","Address_1/ kênh 19/5, Đường 26/3, Phường Bình Hưng Hòa, Bình Tân, Hồ Chí Minh",...,Balcony direction_Tây - Bắc,Balcony direction_Tây - Nam,Balcony direction_Unknown,Balcony direction_Đông,Balcony direction_Đông - Bắc,Balcony direction_Đông - Nam,Legal status_Sale contract,Legal status_Unknown,Furniture state_Full,Furniture state_Unknown
0,84.0,4.5,6.0,4.0,1.0,1.0,8.60,0,0,0,...,0,0,1,0,0,0,0,0,0,1
1,60.0,4.5,6.0,5.0,1.0,1.0,7.50,0,0,0,...,0,0,1,0,0,0,0,1,0,1
2,90.0,6.0,13.0,5.0,1.0,1.0,8.90,0,0,0,...,0,0,0,0,1,0,1,0,0,1
3,54.0,4.5,3.5,2.0,2.0,3.0,5.35,0,0,0,...,0,1,0,0,0,0,0,0,1,0
4,92.0,4.5,6.0,2.0,4.0,4.0,6.90,0,0,0,...,0,0,0,0,0,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24178,60.0,4.0,6.0,2.0,2.0,2.0,3.90,0,0,0,...,0,0,1,0,0,0,0,0,0,0
24179,52.8,8.0,8.0,3.0,3.0,3.0,4.50,0,0,0,...,0,0,1,0,0,0,0,0,0,0
24180,40.2,4.5,6.0,7.0,5.0,6.0,8.90,0,0,0,...,0,0,1,0,0,0,0,1,1,0
24181,40.0,3.5,8.0,4.0,4.0,3.0,5.40,0,0,0,...,0,0,1,0,0,0,0,1,0,1


In [17]:
print(train.isna().sum())
print(train.isnull().sum())

Address              0
Area                 0
Frontage             0
Access Road          0
House direction      0
Balcony direction    0
Floors               0
Bedrooms             0
Bathrooms            0
Legal status         0
Furniture state      0
Price                0
dtype: int64
Address              0
Area                 0
Frontage             0
Access Road          0
House direction      0
Balcony direction    0
Floors               0
Bedrooms             0
Bathrooms            0
Legal status         0
Furniture state      0
Price                0
dtype: int64


## Training

In [18]:
y_train = split_X_y(X_train, y_col_name)
if y_train is None:
    exit()
print(y_train)
display(X_train)

0        8.60
1        7.50
2        8.90
3        5.35
4        6.90
         ... 
24178    3.90
24179    4.50
24180    8.90
24181    5.40
24182    2.80
Name: Price, Length: 24183, dtype: float64


,Area,Frontage,Access Road,Floors,Bedrooms,Bathrooms,"Address_1/ 113, Đường Lâm Thị Hố, Phường Tân Chánh Hiệp, Quận 12, Hồ Chí Minh","Address_1/ Phố Trần Đại Nghĩa, Xã Tân Kiên, Bình Chánh, Hồ Chí Minh","Address_1/ kênh 19/5, Đường 26/3, Phường Bình Hưng Hòa, Bình Tân, Hồ Chí Minh","Address_1/, Đường An Hạ, Xã Phạm Văn Hai, Bình Chánh, Hồ Chí Minh",...,Balcony direction_Tây - Bắc,Balcony direction_Tây - Nam,Balcony direction_Unknown,Balcony direction_Đông,Balcony direction_Đông - Bắc,Balcony direction_Đông - Nam,Legal status_Sale contract,Legal status_Unknown,Furniture state_Full,Furniture state_Unknown
0,84.0,4.5,6.0,4.0,1.0,1.0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,1
1,60.0,4.5,6.0,5.0,1.0,1.0,0,0,0,0,...,0,0,1,0,0,0,0,1,0,1
2,90.0,6.0,13.0,5.0,1.0,1.0,0,0,0,0,...,0,0,0,0,1,0,1,0,0,1
3,54.0,4.5,3.5,2.0,2.0,3.0,0,0,0,0,...,0,1,0,0,0,0,0,0,1,0
4,92.0,4.5,6.0,2.0,4.0,4.0,0,0,0,0,...,0,0,0,0,0,1,0,0,1,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
24178,60.0,4.0,6.0,2.0,2.0,2.0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
24179,52.8,8.0,8.0,3.0,3.0,3.0,0,0,0,0,...,0,0,1,0,0,0,0,0,0,0
24180,40.2,4.5,6.0,7.0,5.0,6.0,0,0,0,0,...,0,0,1,0,0,0,0,1,1,0
24181,40.0,3.5,8.0,4.0,4.0,3.0,0,0,0,0,...,0,0,1,0,0,0,0,1,0,1


In [19]:
print(np.asarray(X_train))

[[84.   4.5  6.  ...  0.   0.   1. ]
 [60.   4.5  6.  ...  1.   0.   1. ]
 [90.   6.  13.  ...  0.   0.   1. ]
 ...
 [40.2  4.5  6.  ...  1.   1.   0. ]
 [40.   3.5  8.  ...  1.   0.   1. ]
 [44.   4.5  6.  ...  1.   0.   1. ]]


In [20]:
model = OLS.OLS()

In [21]:
model.fit(X_train, y_train)

In [22]:
print(model.r2score)

0.8038390595163634


## Evaluation

### Processing test data

In [23]:
fill_null_num_col_median(test, num_col_fill_median)
fill_null_num_col(test, 1, num_col_fill_one)

,Address,Area,Frontage,Access Road,House direction,Balcony direction,Floors,Bedrooms,Bathrooms,Legal status,Furniture state,Price
24183,"Đường Phạm Văn Đồng, Phường Xuân Đỉnh, Bắc Từ ...",37.0,4.5,5.5,NaN,NaN,1.0,1.0,1.0,Have certificate,NaN,6.55
24184,"Đường Bạch Đằng, Phường Tân Lập, Nha Trang, Kh...",29.0,3.7,2.5,Tây - Bắc,NaN,4.0,3.0,3.0,NaN,NaN,2.55
24185,"Dự án HUD Me Linh Central, Xã Thanh Lâm, Mê Li...",117.0,6.0,16.5,NaN,NaN,1.0,1.0,1.0,Have certificate,NaN,7.40
24186,"Dự án Centa Diamond, Đường Hữu Nghị, Xã Phù Ch...",75.0,4.5,5.5,NaN,NaN,4.0,1.0,1.0,NaN,NaN,4.25
24187,"Đường 18B, Phường Bình Hưng Hòa A, Bình Tân, H...",75.0,5.0,5.5,NaN,NaN,4.0,4.0,5.0,Have certificate,NaN,6.35
...,...,...,...,...,...,...,...,...,...,...,...,...
30224,"Đường Lê Quang Định, Phường 1, Gò Vấp, Hồ Chí ...",67.0,4.1,16.0,NaN,NaN,1.0,3.0,2.0,Have certificate,NaN,4.60
30225,"Đường Ngô Gia Tự, Phường Đức Giang, Long Biên,...",30.0,4.5,5.5,NaN,NaN,5.0,3.0,3.0,Have certificate,NaN,4.70
30226,"Đường Gò Dưa, Phường Tam Bình, Thủ Đức, Hồ Chí...",69.4,4.0,15.0,Đông - Bắc,Đông - Bắc,1.0,1.0,1.0,Have certificate,Basic,7.50
30227,"Đường Quang Trung, Phường 11, Gò Vấp, Hồ Chí Minh",96.0,4.5,8.0,NaN,NaN,4.0,1.0,1.0,NaN,NaN,9.50


In [24]:
fill_null_cat_col(test, "Unknown", cat_cols)

X_test = one_hot_encoding(test, cat_cols, True)

display(test)
display(X_test)

,Address,Area,Frontage,Access Road,House direction,Balcony direction,Floors,Bedrooms,Bathrooms,Legal status,Furniture state,Price
24183,"Đường Phạm Văn Đồng, Phường Xuân Đỉnh, Bắc Từ ...",37.0,4.5,5.5,Unknown,Unknown,1.0,1.0,1.0,Have certificate,Unknown,6.55
24184,"Đường Bạch Đằng, Phường Tân Lập, Nha Trang, Kh...",29.0,3.7,2.5,Tây - Bắc,Unknown,4.0,3.0,3.0,Unknown,Unknown,2.55
24185,"Dự án HUD Me Linh Central, Xã Thanh Lâm, Mê Li...",117.0,6.0,16.5,Unknown,Unknown,1.0,1.0,1.0,Have certificate,Unknown,7.40
24186,"Dự án Centa Diamond, Đường Hữu Nghị, Xã Phù Ch...",75.0,4.5,5.5,Unknown,Unknown,4.0,1.0,1.0,Unknown,Unknown,4.25
24187,"Đường 18B, Phường Bình Hưng Hòa A, Bình Tân, H...",75.0,5.0,5.5,Unknown,Unknown,4.0,4.0,5.0,Have certificate,Unknown,6.35
...,...,...,...,...,...,...,...,...,...,...,...,...
30224,"Đường Lê Quang Định, Phường 1, Gò Vấp, Hồ Chí ...",67.0,4.1,16.0,Unknown,Unknown,1.0,3.0,2.0,Have certificate,Unknown,4.60
30225,"Đường Ngô Gia Tự, Phường Đức Giang, Long Biên,...",30.0,4.5,5.5,Unknown,Unknown,5.0,3.0,3.0,Have certificate,Unknown,4.70
30226,"Đường Gò Dưa, Phường Tam Bình, Thủ Đức, Hồ Chí...",69.4,4.0,15.0,Đông - Bắc,Đông - Bắc,1.0,1.0,1.0,Have certificate,Basic,7.50
30227,"Đường Quang Trung, Phường 11, Gò Vấp, Hồ Chí Minh",96.0,4.5,8.0,Unknown,Unknown,4.0,1.0,1.0,Unknown,Unknown,9.50


,Area,Frontage,Access Road,Floors,Bedrooms,Bathrooms,Price,"Address_1 /, Đường Nguyễn Văn Khối, Phường 8, Gò Vấp, Hồ Chí Minh","Address_1/, Phường Bình Hưng Hòa A, Bình Tân, Hồ Chí Minh","Address_1/, Đường Bình Thành, Phường Bình Hưng Hòa B, Bình Tân, Hồ Chí Minh",...,Balcony direction_Tây - Bắc,Balcony direction_Tây - Nam,Balcony direction_Unknown,Balcony direction_Đông,Balcony direction_Đông - Bắc,Balcony direction_Đông - Nam,Legal status_Sale contract,Legal status_Unknown,Furniture state_Full,Furniture state_Unknown
24183,37.0,4.5,5.5,1.0,1.0,1.0,6.55,0,0,0,...,0,0,1,0,0,0,0,0,0,1
24184,29.0,3.7,2.5,4.0,3.0,3.0,2.55,0,0,0,...,0,0,1,0,0,0,0,1,0,1
24185,117.0,6.0,16.5,1.0,1.0,1.0,7.40,0,0,0,...,0,0,1,0,0,0,0,0,0,1
24186,75.0,4.5,5.5,4.0,1.0,1.0,4.25,0,0,0,...,0,0,1,0,0,0,0,1,0,1
24187,75.0,5.0,5.5,4.0,4.0,5.0,6.35,0,0,0,...,0,0,1,0,0,0,0,0,0,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
30224,67.0,4.1,16.0,1.0,3.0,2.0,4.60,0,0,0,...,0,0,1,0,0,0,0,0,0,1
30225,30.0,4.5,5.5,5.0,3.0,3.0,4.70,0,0,0,...,0,0,1,0,0,0,0,0,0,1
30226,69.4,4.0,15.0,1.0,1.0,1.0,7.50,0,0,0,...,0,0,0,0,1,0,0,0,0,0
30227,96.0,4.5,8.0,4.0,1.0,1.0,9.50,0,0,0,...,0,0,1,0,0,0,0,1,0,1


In [25]:
print(test.isna().sum())
print(test.isnull().sum())

Address              0
Area                 0
Frontage             0
Access Road          0
House direction      0
Balcony direction    0
Floors               0
Bedrooms             0
Bathrooms            0
Legal status         0
Furniture state      0
Price                0
dtype: int64
Address              0
Area                 0
Frontage             0
Access Road          0
House direction      0
Balcony direction    0
Floors               0
Bedrooms             0
Bathrooms            0
Legal status         0
Furniture state      0
Price                0
dtype: int64


In [26]:
y_test = split_X_y(X_test, y_col_name)
if y_test is None:
    exit()
print(y_test)

24183    6.55
24184    2.55
24185    7.40
24186    4.25
24187    6.35
         ... 
30224    4.60
30225    4.70
30226    7.50
30227    9.50
30228    3.15
Name: Price, Length: 6046, dtype: float64


In [27]:
model.predict(X_test)

ValueError: shapes (6046,3555) and (8960,) not aligned: 3555 (dim 1) != 8960 (dim 0)